# 01 — Extracción inicial desde SQL Server para exportar datos GIS

Este notebook inicia el proceso de trabajo hacia PostGIS con un alcance **controlado**: el primer paso será la red de media tensión de Montecarlo.

Fuentes priorizadas para esta etapa:

- CAD/DXF: `dxf/Montecarlo_MT.dxf` y su fuente DWG `dwg/Montecarlo_MT.dwg`.
- SQL Server: `TMP_SHAPEFILE`, `Objetos_Red`, `SET`, `seccionadores`, `MT_cables`.

Quedan fuera de la primera geometría: `Trafos`, BT, suministros, alumbrado, reclamos, catastro completo, usuarios, vehículos, seguridad y tablas temporales amplias. Esos dominios pueden analizarse en fases posteriores, pero no deben mezclar el piloto inicial.


## 1. Preparación del entorno

Primero ubicamos la raíz del proyecto y validamos que estén disponibles los datos recibidos. Mantener rutas claras evita que las celdas dependan de la carpeta desde la que se abrió Jupyter.


In [12]:
# pathlib permite trabajar con rutas de archivos de forma portable.
from pathlib import Path

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
# No dependemos del nombre de la carpeta para que funcione aunque el repo se renombre.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Definimos las rutas principales que se usan durante la extracción.
SQLSERVER_DIR = RAIZ / 'sqlserver'
BACKUP_ORIGINAL = SQLSERVER_DIR / '20211020'
EXPORTS = SQLSERVER_DIR / 'exports'

# Mostramos una verificación simple para que el estudiante confirme el contexto.
print('Raíz del proyecto:', RAIZ)
print('Backup recibido existe:', BACKUP_ORIGINAL.exists())
print('Carpeta de exports:', EXPORTS)


Raíz del proyecto: /home/martin/Code/gis_ceml
Backup recibido existe: True
Carpeta de exports: /home/martin/Code/gis_ceml/sqlserver/exports


## 2. Inspección del archivo `20211020`

Antes de restaurar la base no asumimos el formato exacto del archivo. Primero inspeccionamos sus primeros bytes y luego SQL Server leerá la metadata real del backup.


In [13]:
# Abrimos el archivo en modo binario porque todavía no sabemos su estructura interna.
# Leer solo los primeros bytes alcanza para observar la cabecera sin cargar todo el archivo.
if BACKUP_ORIGINAL.exists():
    with BACKUP_ORIGINAL.open('rb') as archivo:
        cabecera = archivo.read(64)
    print('Primeros bytes del archivo recibido:')
    print(cabecera)
else:
    print('No se encontró el archivo sqlserver/20211020. Revisar la carpeta de datos recibidos.')


Primeros bytes del archivo recibido:
b'TAPE\x00\x00\x03\x00\x8c\x00\x0e\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\x00\x87\x059v\xdb<\x05\x00\x00\x00\x01\x00\x00\x00'


## 3. Preparar el backup para el contenedor

El `docker-compose.yml` monta `sqlserver/backup` dentro del contenedor como `/var/opt/mssql/backup`. Para que SQL Server pueda ver el backup, copiamos el archivo recibido a esa carpeta con un nombre estable.


In [14]:
# Ejecutamos el paso de preparación desde el script del laboratorio.
# El script copia sqlserver/20211020 a sqlserver/backup/ceml_gis.bak si hace falta.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh prepare-backup


✅ Backup ya preparado en: sqlserver/backup/ceml_gis.bak


## 4. Levantar SQL Server con Docker

Para inspeccionar el backup necesitamos una instancia de SQL Server. En este laboratorio se define un `docker-compose.yml` en la raíz del proyecto.


### 4.1 Revisar el `docker-compose.yml`

Antes de levantar servicios conviene mirar qué va a crear Docker. Esta celda solo muestra la configuración, no ejecuta el servicio.


In [15]:
# Leemos el docker-compose.yml para explicar en clase qué servicio se va a levantar.
# Esta lectura es segura porque no modifica archivos ni contenedores.
compose_path = RAIZ / 'docker-compose.yml'
print(compose_path.read_text(encoding='utf-8'))


services:
  sqlserver:
    image: mcr.microsoft.com/mssql/server:2019-latest
    container_name: ceml-sqlserver
    environment:
      ACCEPT_EULA: "Y"
      MSSQL_PID: "Express"
      MSSQL_SA_PASSWORD: "${MSSQL_SA_PASSWORD:-CEML_Admin_2026!}"
    ports:
      - "${SQLSERVER_PORT:-1433}:1433"
    volumes:
      - ./sqlserver/data:/var/opt/mssql/data
      - ./sqlserver/backup:/var/opt/mssql/backup
      - ./sqlserver/exports:/exports
    healthcheck:
      test:
        [
          "CMD-SHELL",
          "if [ -x /opt/mssql-tools18/bin/sqlcmd ]; then /opt/mssql-tools18/bin/sqlcmd -C -S localhost -U SA -P \"$$MSSQL_SA_PASSWORD\" -Q \"SELECT 1\" -b -o /dev/null; else /opt/mssql-tools/bin/sqlcmd -S localhost -U SA -P \"$$MSSQL_SA_PASSWORD\" -Q \"SELECT 1\" -b -o /dev/null; fi",
        ]
      interval: 10s
      timeout: 5s
      retries: 20
      start_period: 25s

  postgis:
    image: postgis/postgis:16-3.4
    container_name: ceml-postgis
    environment:
      POSTGRES_DB: "${POSTG

### 4.2 Revisar el script auxiliar

El script `scripts/gis_sqlserver.sh` encapsula comandos largos de Docker y SQL Server. Lo usamos para que el notebook sea legible, pero exportaremos tablas puntuales de MT en vez de usar una exportación GIS amplia.

En esta práctica, los archivos TSV (*Tab-Separated Values*) son tablas de texto donde las columnas se separan con tabuladores. Se usan en lugar de CSV porque algunos textos exportados pueden contener comas y eso complicaría la lectura de columnas.


In [16]:
# Mostramos la ayuda del script para conocer los comandos disponibles.
# El "|| true" evita que Jupyter marque error porque el script sin comando termina mostrando ayuda.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh || true


Uso:
  ./scripts/gis_sqlserver.sh prepare-backup
  ./scripts/gis_sqlserver.sh up
  ./scripts/gis_sqlserver.sh down
  ./scripts/gis_sqlserver.sh info-backup [ruta_backup]
  ./scripts/gis_sqlserver.sh restore [ruta_backup] [db_name] [logical_data_name] [logical_log_name]
  ./scripts/gis_sqlserver.sh restore-auto [ruta_backup] [db_name]
  ./scripts/gis_sqlserver.sh list-tables [db_name]
  ./scripts/gis_sqlserver.sh export-table [db_name] [schema] [table] [output_csv]
  ./scripts/gis_sqlserver.sh export-candidates [db_name]

Notas:
- ruta_backup por defecto: /var/opt/mssql/backup/ceml_gis.bak
- db_name por defecto: CEML_GIS
- output_csv se escribe dentro de /exports del contenedor (montado en sqlserver/exports)
- prepare-backup copia sqlserver/20211020 a sqlserver/backup/ceml_gis.bak
- restore-auto lee los logical names del backup y ejecuta restore automáticamente
- export-table genera archivos con encabezados; para texto libre se recomienda extensión .tsv
- export-candidates exporta las t

### 4.3 Ejecutar el servicio SQL Server

Esta celda puede tardar si Docker necesita descargar la imagen `mcr.microsoft.com/mssql/server:2019-latest`. La contraseña se pasa como variable de entorno para mantener el flujo reproducible.


In [17]:
# Levantamos SQL Server con Docker usando el script del proyecto.
# El script carga .env desde la raíz del proyecto si existe.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh up


chmod: cambiando los permisos de '/home/martin/Code/gis_ceml/sqlserver/data': Operación no permitida
chmod: cambiando los permisos de '/home/martin/Code/gis_ceml/sqlserver/backup': Operación no permitida
chmod: cambiando los permisos de '/home/martin/Code/gis_ceml/sqlserver/exports': Operación no permitida
[+] up 4/5
 ✔ Container ceml-sqlserver Running                                         0.0s
 ⠋ Container ceml-postgis   Waiting                                         0.1s
 ✔ Container ceml-pgadmin   Running                                         0.0s
 ✔ Container ceml-geoserver Running                                         0.0s
 ✔ Container ceml-webgis    Running                                         0.0s
[+] up 4/5
 ✔ Container ceml-sqlserver Running                                         0.0s
 ⠙ Container ceml-postgis   Waiting                                         0.2s
 ✔ Container ceml-pgadmin   Running                                         0.0s
 ✔ Container ceml-geos

## 5. Inspeccionar logical names del backup

Antes de restaurar, SQL Server necesita conocer los nombres lógicos del archivo de datos y del log. `RESTORE FILELISTONLY` lee esa metadata sin restaurar la base.


In [18]:
# RESTORE FILELISTONLY no restaura la base: solo lee metadata interna del backup.
# De este resultado necesitamos los LogicalName de datos y log si se hace restauración manual.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh info-backup


LogicalName PhysicalName Type FileGroupName Size MaxSize FileId CreateLSN DropLSN UniqueId ReadOnlyLSN ReadWriteLSN BackupSizeInBytes SourceBlockSize FileGroupId LogGroupGUID DifferentialBaseLSN DifferentialBaseGUID IsReadOnly IsPresent TDEThumbprint SnapshotUrl
----------- ------------ ---- ------------- ---- ------- ------ --------- ------- -------- ----------- ------------ ----------------- --------------- ----------- ------------ ------------------- -------------------- ---------- --------- ------------- -----------
TecnoGIS C:\Archivos de programa\Microsoft SQL Server\MSSQL.1\MSSQL\DATA\TecnoGIS.mdf D PRIMARY 573505536 35184372080640 1 0 0 C17B0A20-2974-415B-843B-586538780944 0 0 572063744 512 1 NULL 52514000000007400219 809DD3D9-7D85-4365-A725-545731483E99 0 1 NULL NULL
TecnoGIS_log C:\Archivos de programa\Microsoft SQL Server\MSSQL.1\MSSQL\DATA\TecnoGIS_log.ldf L NULL 115081216 2199023255552 2 0 0 A0C235E3-24B8-453E-B58A-3B59E140AE92 0 0 0 512 0 NULL 0 00000000-0000-0000-0000-00

## 6. Restaurar la base

Con la metadata disponible, restauramos la base `CEML_GIS`. El comando automático detecta los logical names y ejecuta el restore con rutas internas del contenedor.


In [19]:
# Restauramos la base de datos de forma reproducible desde la notebook.
# restore-auto detecta logical names y ejecuta RESTORE DATABASE con los archivos correctos.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh restore-auto


✅ Logical data detectado: TecnoGIS
✅ Logical log detectado: TecnoGIS_log
10 percent processed.
20 percent processed.
30 percent processed.
40 percent processed.
50 percent processed.
60 percent processed.
70 percent processed.
80 percent processed.
90 percent processed.
100 percent processed.
Processed 69832 pages for database 'CEML_GIS', file 'TecnoGIS' on file 1.
Processed 3 pages for database 'CEML_GIS', file 'TecnoGIS_log' on file 1.
Converting database 'CEML_GIS' from version 611 to the current version 904.
Database 'CEML_GIS' running the upgrade step from version 611 to version 621.
Database 'CEML_GIS' running the upgrade step from version 621 to version 622.
Database 'CEML_GIS' running the upgrade step from version 622 to version 625.
Database 'CEML_GIS' running the upgrade step from version 625 to version 626.
Database 'CEML_GIS' running the upgrade step from version 626 to version 627.
Database 'CEML_GIS' running the upgrade step from version 627 to version 628.
Database 'CEML

## 7. Listar tablas disponibles

Una vez restaurada la base, listamos las tablas para verificar que existan las cinco piezas de MT: `TMP_SHAPEFILE`, `Objetos_Red`, `SET`, `seccionadores` y `MT_cables`.


In [20]:
# Listamos tablas de la base restaurada para confirmar el inventario disponible.
# Este listado ayuda a verificar nombres exactos antes de exportar tablas del piloto.
!cd "{RAIZ}" && ./scripts/gis_sqlserver.sh list-tables CEML_GIS


TABLE_SCHEMA TABLE_NAME
------------ ----------
dbo __RCL_RECLAMOS
dbo __RCL_RECLAMOS_DESCRIPCION
dbo _ERR_SET
dbo ABONADOS_TEL
dbo alturas
dbo AP_ALUMBRADO
dbo AP_alumbrado_luminarias
dbo AP_Habilitaciones
dbo AP_mantenimientos
dbo AP_Proyectos
dbo BT_cables
dbo BT_cables_cat
dbo ca_calles
dbo CA_Otros
dbo cad_auxiliar
dbo CAL_causa
dbo CAL_cortados_0
dbo CAL_cortados_12
dbo CAL_cortados_15
dbo CAL_cortados_16
dbo CAL_cortados_19
dbo CAL_cortes_0
dbo CAL_cortes_12
dbo CAL_cortes_15
dbo CAL_cortes_16
dbo CAL_cortes_19
dbo CAL_curva1
dbo CAL_curva10
dbo CAL_curva11
dbo CAL_curva12
dbo CAL_curva13
dbo CAL_curva14
dbo CAL_curva2
dbo CAL_curva3
dbo CAL_curva4
dbo CAL_curva5
dbo CAL_curva6
dbo CAL_curva7
dbo CAL_curva8
dbo CAL_curva9
dbo CAL_guardia
dbo CAL_mensual
dbo CAL_motivo
dbo CAL_MultasXTarifas
dbo CAL_semestral
dbo CAL_semestres
dbo CAL_turnos_guardia
dbo CAL_valores
dbo CalidadTecnica
dbo Capas
dbo CapasImg
dbo catastro
dbo consumo_crédito
dbo consumo_prepago
dbo Consumos_Aguas
db

## 8. Exportar tablas de MT

En esta primera etapa exportamos solo las tablas necesarias para estudiar la red MT Montecarlo:

- `TMP_SHAPEFILE`: contiene `Datos_Objeto`, texto de estilo DXF separado por tokens `§`; desde allí se puede reconstruir geometría.
- `Objetos_Red`: aporta atributos de red y se cruza con `TMP_SHAPEFILE` por `idcad/IDCAD` y `coop/COOP` normalizados.
- `SET`, `seccionadores`, `MT_cables`: tablas de dominio/atributos para enriquecer entidades MT.

Reglas de reconstrucción previstas para `Datos_Objeto`:

- `INSERT` y `CIRCLE` se tratan como `POINT`.
- `LINE` y `LWPOLYLINE` abierta se tratan como `LINESTRING`.
- `LWPOLYLINE` cerrada se trata como `POLYGON`.

`Trafos`, BT, suministros, alumbrado, reclamos y catastro quedan documentados como fases posteriores, no como geometría inicial.


In [21]:
# Exportamos explícitamente las tablas del piloto MT Montecarlo.
# Usamos export-table para evitar una exportación amplia que mezcle dominios fuera de alcance.
tablas_piloto_mt = [
    ('dbo', 'TMP_SHAPEFILE', 'tmp_shapefile.tsv'),
    ('dbo', 'Objetos_Red', 'objetos_red.tsv'),
    ('dbo', 'SET', 'set.tsv'),
    ('dbo', 'seccionadores', 'seccionadores.tsv'),
    ('dbo', 'MT_cables', 'mt_cables.tsv'),
]

# Ejecutamos un comando por tabla para que sea claro qué se exporta y con qué nombre queda el TSV.
for esquema, tabla, salida in tablas_piloto_mt:
    print(f'Exportando {esquema}.{tabla} hacia sqlserver/exports/{salida}')
    !cd "{RAIZ}" && ./scripts/gis_sqlserver.sh export-table CEML_GIS {esquema} {tabla} {salida}


Exportando dbo.TMP_SHAPEFILE hacia sqlserver/exports/tmp_shapefile.tsv
✅ Export generado en: sqlserver/exports/tmp_shapefile.tsv
Exportando dbo.Objetos_Red hacia sqlserver/exports/objetos_red.tsv
✅ Export generado en: sqlserver/exports/objetos_red.tsv
Exportando dbo.SET hacia sqlserver/exports/set.tsv
✅ Export generado en: sqlserver/exports/set.tsv
Exportando dbo.seccionadores hacia sqlserver/exports/seccionadores.tsv
✅ Export generado en: sqlserver/exports/seccionadores.tsv
Exportando dbo.MT_cables hacia sqlserver/exports/mt_cables.tsv
✅ Export generado en: sqlserver/exports/mt_cables.tsv


## 9. Cierre de la extracción

En esta notebook se preparó el backup, se levantó SQL Server, se restauró la base `CEML_GIS` y se exportaron los TSV del piloto MT hacia `sqlserver/exports/`.

- Quedaron exportadas las tablas `TMP_SHAPEFILE`, `Objetos_Red`, `SET`, `seccionadores` y `MT_cables`.
- La notebook 02 analiza esos archivos para validar claves, contenido de `Datos_Objeto` y cobertura de unión.
